# IAM-811 — Auto-etiquetado con YOLOE, Subida a Roboflow y Entrenamiento de M2 (`YOLO11s-seg` — 5 Clases)
**Estudiante:** Amilkar Hugo Guarachi Nina · **Docente:** Ing. Alvaro Pinto Mamani  
**Clases oficiales del peaje:** `0: moto`, `1: auto`, `2: minibus`, `3: camion`, `4: bus`

Este notebook sigue exactamente la misma estructura de la **Celda 8 del notebook del docente**:
1. **Celda 1:** Instala Ultralytics y CLIP para `YOLOE`.
2. **Celda 2 (Imágenes):** Subes `mis_imagenes_vehiculos.zip` (y opcionalmente el `.zip` de `minibuses-la-paz-benlc` descargado de Roboflow Universe).
3. **Celda 3 (Auto-etiquetado Real con YOLOE-seg):** Usa la red neuronal `yoloe-11l-seg.pt` para segmentar automáticamente las máscaras poligonales reales de las 5 clases (`moto, auto, minibus, camion, bus`), crea `dataset_m2_vehiculos.zip` y te lo descarga para que lo subas a tu proyecto de **Roboflow (Instance Segmentation)**.
4. **Celda 4 (Entrenamiento M2):** Entrena `yolo11s-seg.pt` (ya sea desde la carpeta generada o desde tu API Key de Roboflow), imprime las métricas reales (`mAP50`, `mAP50-95`) y descarga automáticamente **`m2_vehiculos_seg.pt`**.

In [ ]:
# 1. Instalación y diagnóstico del entorno en Google Colab
!nvidia-smi -L
!pip -q install -U ultralytics roboflow
!pip -q install 'git+https://github.com/ultralytics/CLIP.git'

import os, glob, random, shutil, cv2, numpy as np, torch
from ultralytics import YOLO, YOLOE
from google.colab import files
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo activo:', DEVICE)

In [ ]:
# 2. Subir archivo(s) .zip con imágenes de vehículos (ej. mis_imagenes_vehiculos.zip + minibuses-la-paz.zip)
DIR_IMGS = '/content/imagenes_m2'
shutil.rmtree(DIR_IMGS, ignore_errors=True)
os.makedirs(DIR_IMGS, exist_ok=True)

print('Sube tu archivo mis_imagenes_vehiculos.zip (y cualquier otro .zip de imágenes de vehículos):')
for nombre in files.upload().keys():
    if nombre.lower().endswith('.zip'):
        !unzip -qo "$nombre" -d /content/imagenes_m2
    else:
        shutil.move(nombre, DIR_IMGS)

EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
LISTA_IMGS = sorted(p for p in glob.glob(DIR_IMGS + '/**/*', recursive=True) if p.lower().endswith(EXT))
print(f'Total de imágenes cargadas: {len(LISTA_IMGS)}')

In [ ]:
# 3. Auto-etiquetado de máscaras reales en formato YOLO-seg con YOLOE (5 clases: moto, auto, minibus, camion, bus)
PROMPTS_M2 = {
    'motorcycle': 'moto', 'motorbike': 'moto',
    'car': 'auto', 'sedan': 'auto', 'suv': 'auto',
    'minibus': 'minibus', 'van': 'minibus', 'microbus': 'minibus',
    'truck': 'camion', 'cargo truck': 'camion',
    'bus': 'bus'
}
ID_CLASE = {'moto': 0, 'auto': 1, 'minibus': 2, 'camion': 3, 'bus': 4}

nombres_p = list(PROMPTS_M2.keys())
modelo_yoloe = YOLOE('yoloe-11l-seg.pt')
modelo_yoloe.set_classes(nombres_p, modelo_yoloe.get_text_pe(nombres_p))

DIR_DS_M2 = '/content/dataset_m2_vehiculos'
shutil.rmtree(DIR_DS_M2, ignore_errors=True)
for s in ('train', 'val'):
    os.makedirs(f'{DIR_DS_M2}/images/{s}', exist_ok=True)
    os.makedirs(f'{DIR_DS_M2}/labels/{s}', exist_ok=True)

random.seed(42)
imgs = LISTA_IMGS[:]
random.shuffle(imgs)
n_val = max(2, int(len(imgs) * 0.2))
total_instancias = 0

for i, ruta in enumerate(imgs):
    split = 'val' if i < n_val else 'train'
    im = cv2.imread(ruta)
    if im is None:
        continue
    H, W = im.shape[:2]
    r = modelo_yoloe.predict(im, conf=0.22, device=DEVICE, agnostic_nms=True, verbose=False)[0]
    lineas = []
    if r.masks is not None and len(r.boxes):
        for poly, c in zip(r.masks.xy, r.boxes.cls.cpu().numpy().astype(int)):
            clase = PROMPTS_M2.get(r.names[c])
            if clase is None or len(poly) < 3:
                continue
            coords = ' '.join(f'{min(max(x/W, 0.0), 1.0):.5f} {min(max(y/H, 0.0), 1.0):.5f}' for x, y in poly)
            lineas.append(f'{ID_CLASE[clase]} {coords}')
    base = f'{i:05d}_' + os.path.splitext(os.path.basename(ruta))[0]
    cv2.imwrite(f'{DIR_DS_M2}/images/{split}/{base}.jpg', im)
    with open(f'{DIR_DS_M2}/labels/{split}/{base}.txt', 'w') as f:
        f.write('\n'.join(lineas))
    total_instancias += len(lineas)

DATA_YAML_M2 = f'{DIR_DS_M2}/data.yaml'
with open(DATA_YAML_M2, 'w') as f:
    f.write(f'path: {DIR_DS_M2}\ntrain: images/train\nval: images/val\nnc: 5\nnames: [moto, auto, minibus, camion, bus]\n')

print(f'Vehículos segmentados y etiquetados: {total_instancias}')
!cd /content && zip -qr dataset_m2_vehiculos.zip dataset_m2_vehiculos
files.download('/content/dataset_m2_vehiculos.zip')  # Súbelo a Roboflow Instance Segmentation para obtener el link del README

In [ ]:
# 4. Entrenar M2 (YOLO11s-seg) y descargar m2_vehiculos_seg.pt
# Si ya lo subiste a Roboflow y quieres descargarlo desde tu workspace, pon USAR_ROBOFLOW = True
USAR_ROBOFLOW = False
if USAR_ROBOFLOW:
    from roboflow import Roboflow
    rf = Roboflow(api_key='TU_ROBOFLOW_API_KEY')
    ds = rf.workspace('TU_WORKSPACE').project('TU_PROYECTO_M2').version(1).download('yolov11')
    DATA_YAML_M2 = ds.location + '/data.yaml'

modelo_m2 = YOLO('yolo11s-seg.pt')
modelo_m2.train(
    data=DATA_YAML_M2,
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs/segment',
    name='vehiculos_m2',
    exist_ok=True
)
val_m2 = YOLO('/content/runs/segment/vehiculos_m2/weights/best.pt').val(data=DATA_YAML_M2)
print('====================================')
print('MÉTRICAS REALES M2 PARA EL INFORME:')
print('mAP50 (Mask)    :', round(float(val_m2.seg.map50), 4))
print('mAP50-95 (Mask) :', round(float(val_m2.seg.map), 4))
print('mAP50 (Box)     :', round(float(val_m2.box.map50), 4))
print('====================================')
shutil.copy('/content/runs/segment/vehiculos_m2/weights/best.pt', '/content/m2_vehiculos_seg.pt')
files.download('/content/m2_vehiculos_seg.pt')